# Обработка праймеров — лёгкие цепи мыши

Четыре предположительных мотива 5′-границы IGL, приведённых в статье и выведенных авторами по первым 30 нуклеотидам R1 (`IGLV1`, `IGLC_var1`, `IGLC2`, `IGLC3`), удаляются из R1 командой `MaskPrimers align --mode cut`. Последовательности проприетарных праймеров набора SMARTer и точные координаты отжига не опубликованы. Точные последовательности IGK-праймеров набора также неизвестны, поэтому риды без совпадения с праймером сохраняются в отдельной ветке. Обе ветки остаются парными и передаются в pRESTO.


In [ ]:
from pathlib import Path
import gzip, json, os, shutil, subprocess, time
REPO=Path.cwd().resolve()
if not (REPO/'notebooks').is_dir(): REPO=REPO.parent
VOLUME=Path(os.environ.get('BCR_VOLUME','/data/user/epishkin'))
if not (VOLUME/'raw').is_dir(): VOLUME=REPO
ENV=Path(os.environ.get('BCR_ENV','/opt/conda/envs/bcr_env'))
if not ENV.is_dir(): ENV=Path('/Users/epishkin/mamba/envs/bcr_env')
RUN='SRR32426580'; RAW=VOLUME/'raw'/'PRJNA1226555'
ROOT=VOLUME/'results'/'PRJNA1226555'/'branches'/'legacy_qtrim_min250'
R1=RAW/f'{RUN}_1.fastq.gz'; R2=RAW/f'{RUN}_2.fastq.gz'
def tool(name):
    p=ENV/'bin'/name
    if p.is_file(): return p
    q=shutil.which(name)
    if q: return Path(q)
    raise FileNotFoundError(name)
def fqcount(path):
    with gzip.open(path,'rt') as h: n=sum(1 for _ in h)
    assert n%4==0,path
    return n//4
def run(cmd,out,err,outputs=(),heartbeat=30):
    out=Path(out); err=Path(err); out.parent.mkdir(parents=True,exist_ok=True)
    started=time.monotonic()
    with out.open('w') as o,err.open('w') as e:
        p=subprocess.Popen([str(x) for x in cmd],stdout=o,stderr=e,text=True)
        print(f'PID={p.pid}',flush=True)
        while p.poll() is None:
            sizes=' '.join(f'{Path(x).name}={Path(x).stat().st_size/1e6:.1f}MB' for x in outputs if Path(x).exists())
            print(f'PID={p.pid} elapsed={(time.monotonic()-started)/60:.1f}min {sizes}',flush=True)
            time.sleep(heartbeat)
    if p.returncode: raise RuntimeError(f'rc={p.returncode}; see {err}')
    print(f'DONE elapsed={(time.monotonic()-started)/60:.1f}min',flush=True)
for p in (R1,R2): assert p.is_file(),p
print('ROOT',ROOT)


In [ ]:
IGL_PRIMERS={'IGL_1':'AGCTCTTCAGAGGAAGGTGG','IGL_2':'AGCTCTTCAGGGGAAGGTGG','IGL_3':'AGCTCCTCAGAGGAAGGTGG','IGL_4':'AGCTCCTCAGGGGAAGGTGG'}
t1=ROOT/'trimmed'/'fastq'/f'{RUN}_1.trim.fastq.gz'; t2=ROOT/'trimmed'/'fastq'/f'{RUN}_2.trim.fastq.gz'
for p in (t1,t2): assert p.is_file(),p
BASE=ROOT/'pr_trimmed'; MASK=BASE/'mask'; SYNC=BASE/'sync'; LOG=BASE/'logs'
shutil.rmtree(BASE,ignore_errors=True)
for d in (MASK,SYNC,LOG): d.mkdir(parents=True)
primers=BASE/'mouse_light_igl_primers.fasta'
primers.write_text(''.join('>'+k+chr(10)+v+chr(10) for k,v in IGL_PRIMERS.items()))
run([tool('MaskPrimers.py'),'align','-s',t1,'-p',primers,'--mode','cut','--maxerror','0.2','--maxlen','50','--nproc','4','--failed','--gzip-output','--outdir',MASK,'--outname',f'{RUN}_R1_IGL'],LOG/'mask.stdout.log',LOG/'mask.stderr.log',list(MASK.glob('*')))
pass_r1=next(MASK.glob('*primers-pass.fastq.gz')); fail_r1=next(MASK.glob('*primers-fail.fastq.gz'))
assert fqcount(pass_r1)+fqcount(fail_r1)==fqcount(t1)
print('MASK_PASS',fqcount(pass_r1),'MASK_FAIL',fqcount(fail_r1))


In [ ]:
BASE=ROOT/'pr_trimmed'; MASK=BASE/'mask'; SYNC=BASE/'sync'; LOG=BASE/'logs'
t1=ROOT/'trimmed'/'fastq'/f'{RUN}_1.trim.fastq.gz'; t2=ROOT/'trimmed'/'fastq'/f'{RUN}_2.trim.fastq.gz'
pass_r1=next(MASK.glob('*primers-pass.fastq.gz')); fail_r1=next(MASK.glob('*primers-fail.fastq.gz'))
def sync_branch(branch,r1_subset):
    out=SYNC/branch; shutil.rmtree(out,ignore_errors=True); out.mkdir()
    run([tool('PairSeq.py'),'-1',r1_subset,'-2',t2,'--coord','sra','--gzip-output','--outdir',out,'--outname',f'{RUN}_{branch}'],LOG/f'pair_{branch}.stdout.log',LOG/f'pair_{branch}.stderr.log',list(out.glob('*')))
    files=sorted(out.glob('*pair-pass*.fastq.gz')); assert len(files)==2,files
    r1=next(p for p in files if '-1_pair-pass' in p.name)
    r2=next(p for p in files if '-2_pair-pass' in p.name)
    assert fqcount(r1)==fqcount(r2)==fqcount(r1_subset)
    return r1,r2
pass_pair=sync_branch('igl_primer_pass',pass_r1)
fail_pair=sync_branch('primer_unmatched',fail_r1)
summary={'trimmed_pairs':fqcount(t1),'igl_primer_pass_pairs':fqcount(pass_pair[0]),'primer_unmatched_pairs':fqcount(fail_pair[0])}
(BASE/'primer_summary.json').write_text(json.dumps(summary,indent=2)); print(summary)
